## Quiet the logs

Disable DEBUG logging to avoid noisy output from redis-py, httpx, etc.

In [1]:
import logging

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
logging.getLogger().setLevel(logging.WARNING)
for _noisy in ("redis", "httpx", "httpx2", "httpcore", "httpcore2",
               "urllib3", "asyncio", "mcp"):
    logging.getLogger(_noisy).setLevel(logging.WARNING)

print("log level: WARNING (set INFO or DEBUG here if you want the detail back)")

log level: WARNING (set INFO or DEBUG here if you want the detail back)


## One-sample t-test across the fleet

t also scales with sqrt(n), so a t computed over the wrong row count is
self-consistent and still wrong. Watch n as closely as t.


## Parameters

In [2]:
import os, time, json, subprocess

HEAD_APP_ID   = os.environ.get("HEAD_APP_ID", "scarlet-agent-head")
WORKER_APP_ID = os.environ.get("WORKER_APP_ID", "scarlet-agent-worker")

HEAD_NODE    = os.environ.get("HEAD_NODE_ADDRESS", "head")
WORKER_NODES = [n.strip() for n in
                os.environ.get("WORKER_NODE_ADDRESSES", "worker1,worker2,worker3,worker4").split(",")
                if n.strip()]

HEAD_BUS      = os.environ.get("HEAD_BUS", f"{HEAD_APP_ID}_headagent")
HEAD_HOST     = os.environ.get("HEAD_HOST", "agent-head")
HEAD_MCP_PORT = os.environ.get("HEAD_MCP_PORT", "8090")

_explicit_ids = os.environ.get("EXPECTED_AGENT_IDS", "").strip()
if _explicit_ids:
    EXPECTED_AGENT_IDS = {i.strip() for i in _explicit_ids.split(",") if i.strip()}
    HEAD_AGENT_ID      = f"{HEAD_APP_ID}_{HEAD_NODE}"
    WORKER_AGENT_IDS   = EXPECTED_AGENT_IDS - {HEAD_AGENT_ID}
else:
    HEAD_AGENT_ID      = f"{HEAD_APP_ID}_{HEAD_NODE}"
    WORKER_AGENT_IDS   = {f"{WORKER_APP_ID}_{n}" for n in WORKER_NODES}
    EXPECTED_AGENT_IDS = {HEAD_AGENT_ID} | WORKER_AGENT_IDS

GUSTAVO_MANAGED_AGENTS = os.environ.get("GUSTAVO_MANAGED_AGENTS", "false").lower() == "true"
COMPOSE_FILE = os.environ.get("COMPOSE_FILE_PATH", "/workspace/docker-compose.yml")

DATA_DIR = os.environ.get("DATA_DIR", "/data")

print(f"head app id   : {HEAD_APP_ID}")
print(f"worker app id : {WORKER_APP_ID}")
print(f"workers       : {WORKER_NODES}")
print(f"fleet managed : {'Gustavo' if GUSTAVO_MANAGED_AGENTS else 'this notebook (docker compose)'}")

head app id   : loop-agent-head
worker app id : loop-agent-worker
workers       : ['worker1', 'worker2', 'worker3', 'worker4']
fleet managed : this notebook (docker compose)


## Generate each worker's matrix data and its local config

Four workers, four different shapes, four different column sets. The
intersection is never written down anywhere — it has to be discovered.

In [3]:
import numpy as np
import pandas as pd
import yaml

LAYOUT = {
    "worker1": ("plant_alpha/turbine_vibration.csv",  "turbine_alpha_vibration",  80,
                ["vibration_rms", "bearing_temp_c", "shaft_speed_rpm", "torque_nm", "power_kw"]),
    "worker2": ("feeds/grid_telemetry.csv",           "grid_telemetry_feed",     100,
                ["vibration_rms", "bearing_temp_c", "shaft_speed_rpm", "power_kw"]),
    "worker3": ("archive/2026/q3/unit7_sensors.csv",  "unit7_archive_q3",        120,
                ["vibration_rms", "bearing_temp_c", "shaft_speed_rpm", "torque_nm", "power_kw",
                 "ambient_temp_c"]),
    "worker4": ("raw_drop/sensor_export_FINAL.csv",   "bulk_sensor_drop",         90,
                ["vibration_rms", "shaft_speed_rpm", "power_kw"]),
}

PARAMS = {
    "vibration_rms":   (2.5,    0.6),
    "bearing_temp_c":  (65.0,   8.0),
    "shaft_speed_rpm": (1800.0, 120.0),
    "torque_nm":       (340.0,  45.0),
    "power_kw":        (64.0,   11.0),
    "ambient_temp_c":  (22.0,   4.0),
}

for i, (node, (relpath, source_name, rows, cols)) in enumerate(LAYOUT.items(), start=1):
    rng = np.random.default_rng(seed=i * 1000)
    start = pd.Timestamp("2026-01-01T00:00:00")
    frame = {"timestamp": [start + pd.Timedelta(minutes=k) for k in range(rows)]}
    for c in cols:
        mean, sd = PARAMS[c]
        frame[c] = np.clip(rng.normal(mean, sd, size=rows), 0.0, None)
    df = pd.DataFrame(frame)

    gap_cols = list(rng.permutation(cols))[:3]
    gap_rows = rng.choice(rows, size=3, replace=False)
    for r, c in zip(gap_rows, gap_cols):
        df.at[r, c] = np.nan

    path = os.path.join(DATA_DIR, relpath)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df.to_csv(path, index=False)

    scarlet_dir = os.path.join(DATA_DIR, node, ".scarlet")
    os.makedirs(scarlet_dir, exist_ok=True)
    actual_gaps = sorted(df.columns[df.isna().any()].tolist())
    with open(os.path.join(scarlet_dir, "config.yaml"), "w") as fh:
        yaml.safe_dump({"sources": [{
            "name": source_name, "type": "csv", "mode": "local",
            "description": (f"{rows} rows. Columns: {', '.join(cols)}. "
                            f"gaps in columns: {', '.join(actual_gaps)}"),
            "path": path,
        }]}, fh, sort_keys=False)

    has = "has torque_nm" if "torque_nm" in cols else "NO torque_nm"
    print(f"{source_name:26s} {rows:>4} rows  {has}")

SHARED = ["vibration_rms", "shaft_speed_rpm", "power_kw"]
MU0 = 60.0

turbine_alpha_vibration      80 rows  has torque_nm
grid_telemetry_feed         100 rows  NO torque_nm
unit7_archive_q3            120 rows  has torque_nm
bulk_sensor_drop             90 rows  NO torque_nm


## Ground truth, from the same files the workers read

In [4]:
import pandas as pd

pooled = pd.concat(
    [pd.read_csv(f"{DATA_DIR}/{rel}")[SHARED].dropna(how="any")
     for rel, _, _, _ in LAYOUT.values()],
    ignore_index=True)
n = len(pooled)
print(f"n = {n}")
for c in SHARED:
    xbar = pooled[c].mean()
    s = pooled[c].std(ddof=1)
    t_stat = (xbar - MU0) / (s / np.sqrt(n))
    print(f"  {c:<18} expected t = {t_stat:>16.4f}")

n = 380
  vibration_rms      expected t =       -1870.5760
  shaft_speed_rpm    expected t =         281.8609
  power_kw           expected t =           6.8990


## Ask the fleet

No source, no column, no SQL.

In [5]:
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client


async def ask_scarlet_agent(message: str) -> str:
    url = f"http://{HEAD_HOST}:{HEAD_MCP_PORT}/mcp"
    async with streamable_http_client(url) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            result = await session.call_tool("ask_scarlet_agent", {"message": message})
            return result.content[0].text


answer = await ask_scarlet_agent(f"Run a one-sample t-test across the fleet using the t_test skill with mu0={MU0}. Report t for each column and state n once. Do not name any data source or column yourself.")
print(answer)

The one-sample t-test with μ₀ = 60.0 was run across the fleet. The results are:

- **t-statistics**:
  - `power_kw`: **6.899**
  - `shaft_speed_rpm`: **281.861**
  - `vibration_rms`: **−1870.576**

- **Sample size (n)**: **380**

All p-values are effectively 0 (two-sided), indicating strong evidence against the null hypothesis that the mean equals 60.0 for all three columns.


## Check it

The second column is the believable wrong answer (if sqrt(n) were omitted).

In [6]:
print(f"n = {n}")
for c in SHARED:
    xbar = pooled[c].mean()
    s = pooled[c].std(ddof=1)
    correct_t = (xbar - MU0) / (s / np.sqrt(n))
    wrong_t = (xbar - MU0) / s
    print(f"  {c:<18} correct t={correct_t:>16.4f}   wrong t (no sqrt(n))={wrong_t:>16.4f}")

n = 380
  vibration_rms      correct t=      -1870.5760   wrong t (no sqrt(n))=        -95.9585
  shaft_speed_rpm    correct t=        281.8609   wrong t (no sqrt(n))=         14.4592
  power_kw           correct t=          6.8990   wrong t (no sqrt(n))=          0.3539


## All three tails

`mode` selects which tail the alternative hypothesis lives in, and every
hypothesis test in the harness takes it:

| mode | alternative | p-value |
|---|---|---|
| `two-sided` (default) | the value **differs** from the hypothesised one | `2*min(u, 1-u)` |
| `upper` | it is **greater** | `u` |
| `lower` | it is **less** | `1 - u` |

where `u` is the upper-tail probability of the statistic. Getting this
wrong is quiet: a two-sided p-value used for a one-sided question is
exactly double, and an upper tail used for a two-sided one is exactly
half. Both still look like probabilities.

The expected values below are computed here with scipy, so the fleet's
three answers can be checked against them one at a time.


In [7]:
from scipy import stats

print(f"n = {n}   mu0 = {MU0}   df = {n - 1}")
print(f"{'column':<18} {'t':>10} {'upper':>14} {'lower':>14} {'two-sided':>14}")
for c in SHARED:
    t_stat = (pooled[c].mean() - MU0) / (pooled[c].std(ddof=1) / np.sqrt(n))
    u = stats.t.sf(t_stat, n - 1)
    print(f"{c:<18} {t_stat:>10.4f} {u:>14.6g} {1-u:>14.6g} {2*min(u, 1-u):>14.6g}")


n = 380   mu0 = 60.0   df = 379
column                      t          upper          lower      two-sided
vibration_rms      -1870.5760              1              0              0
shaft_speed_rpm      281.8609              0              1              0
power_kw               6.8990    1.10337e-11              1    2.20674e-11


### Ask the fleet, once per tail

In [8]:
for mode in ("two-sided", "upper", "lower"):
    answer = await ask_scarlet_agent(
        f"Run a one-sample t-test across the fleet using the t_test skill "
        f"with mu0={MU0} and mode='{mode}'. Report the t statistic, the "
        f"degrees of freedom and the p-value for each column, and say "
        f"which mode was used. Do not name any data source or column "
        f"yourself.")
    print(f"===== mode={mode} =====")
    print(answer)
    print()


===== mode=two-sided =====
The one-sample t-test was conducted with mu0=60.0 and mode='two-sided'. Here are the results for each column:

- **power_kw**: t = 6.899, df = 379, p-value = 2.21e-11
- **shaft_speed_rpm**: t = 281.861, df = 379, p-value = 0.0
- **vibration_rms**: t = -1870.576, df = 379, p-value = 0.0

The test used a two-sided alternative hypothesis, meaning we're testing whether the true mean differs from 60.0 in either direction. All columns show statistically significant differences from the hypothesized mean of 60.0 (all p-values < 0.001).



===== mode=upper =====
I have the results from the one-sample t-test. Let me report them clearly:

**Test Details:**
- Test type: One-sample t-test
- Hypothesized mean (μ₀): 60.0
- Alternative hypothesis mode: upper (testing if the true mean is greater than 60.0)
- Degrees of freedom: 379
- Total observations: 380

**Results by column:**

1. **power_kw**:
   - t-statistic: 6.899
   - p-value (upper tail): 1.10 × 10⁻¹¹
   - Interpretation: Strong evidence that the mean power_kw is greater than 60.0

2. **shaft_speed_rpm**:
   - t-statistic: 281.861
   - p-value (upper tail): 0.0
   - Interpretation: Extremely strong evidence that the mean shaft_speed_rpm is greater than 60.0

3. **vibration_rms**:
   - t-statistic: -1870.576
   - p-value (upper tail): 1.0
   - Interpretation: No evidence that the mean vibration_rms is greater than 60.0; in fact, the sample mean is likely less than 60.0

The upper-tail p-values indicate the probability of observing a t-statistic as extreme or more extrem

===== mode=lower =====
The one-sample t-test was conducted with mu0=60.0 and mode='lower' (testing if the true mean is less than 60.0).

Results by column:

1. **power_kw**: 
   - t statistic: 6.899
   - degrees of freedom: 379
   - p-value: 0.9999999999889663

2. **shaft_speed_rpm**: 
   - t statistic: 281.861
   - degrees of freedom: 379
   - p-value: 1.0

3. **vibration_rms**: 
   - t statistic: -1870.576
   - degrees of freedom: 379
   - p-value: 0.0

The mode used was 'lower', meaning the alternative hypothesis is that the true mean is less than 60.0. For the vibration_rms column, the extremely negative t statistic and p-value of 0.0 indicate very strong evidence that the mean is less than 60.0. For the other columns, the positive t statistics and high p-values indicate no evidence that the means are less than 60.0.



## Teardown

In [9]:
if GUSTAVO_MANAGED_AGENTS:
    print("Gustavo owns this fleet - leaving it alone.")
else:
    print("agent containers stopped and removed")

agent containers stopped and removed
